In [14]:
# Task 1 and 2
from pyspark import SparkContext

sc = SparkContext("local[*]", "DataIO")

# Load the CSV file
lines = sc.textFile("sales_data.csv")

# Skip header line
header = lines.first()
data = lines.filter(lambda line: line != header)

print(f"Header: {header}")
print(f"Data records: {data.count()}")
print(f"First record: {data.first()}")


def parse_record(line):
    """Parse CSV line into structured data."""
    parts = line.split(",")
    return {
        "product_id": parts[0],
        "name": parts[1],
        "category": parts[2],
        "price": float(parts[3]),
        "quantity": int(parts[4])
    }

# Parse all records
parsed = data.map(parse_record)

# Show parsed data
for record in parsed.take(3):
    print(record)



Header: product_id,name,category,price,quantity
Data records: 7
First record: P001,Laptop,Electronics,999.99,5
{'product_id': 'P001', 'name': 'Laptop', 'category': 'Electronics', 'price': 999.99, 'quantity': 5}
{'product_id': 'P002', 'name': 'Mouse', 'category': 'Electronics', 'price': 29.99, 'quantity': 50}
{'product_id': 'P003', 'name': 'Desk', 'category': 'Furniture', 'price': 199.99, 'quantity': 10}


In [15]:
import os
import shutil

# Task 3
# Calculate revenue for each product
revenue = parsed.map(lambda r: f"{r['product_id']},{r['name']},{r['price'] * r['quantity']:.2f}")

# Save to output directory
# YOUR CODE: Use saveAsTextFile to save revenue data overriding the existing output directory
revanue_path = "output/revenue_data"
if os.path.exists(revanue_path):
    shutil.rmtree(revanue_path)

revenue.saveAsTextFile(revanue_path)

In [16]:
# YOUR CODE: Create sales_data_2.csv with more records
# YOUR CODE: Load all CSV files using wildcard pattern
all_data = sc.textFile("sales_data*.csv")

In [ ]:
# YOUR CODE: Use coalesce(1) before saveAsTextFile
# This creates a single output file instead of multiple parts
# There is an issue with the output directory if it already exists, so we need to remove it first
sales_data_path = "output/sales_data"
if os.path.exists(sales_data_path):
    shutil.rmtree(sales_data_path)
all_data.coalesce(1).saveAsTextFile(sales_data_path)


In [18]:
sc.stop()